# SF Zipcode-Census Match Checker

This notebook checks whether the original ZIP code matches the ZIP in the Census geocoder matched address.

Note: Census tract and ZIP are different geography systems (many-to-many), so this check uses the geocoder standardized ZIP from `matched_address` as the best consistency check.

In [81]:
import pandas as pd
import re

in_path = "../../data/1_derived/3_sf_properties_census_geocoded.csv"
df = pd.read_csv(in_path)
print(f"Loaded {len(df):,} rows from {in_path}")
print(df.columns.tolist())

Loaded 9,352 rows from ../../data/1_derived/3_sf_properties_census_geocoded.csv
['LeaseCompId', 'PropertyId', 'PropertyName', 'Tenant.Name', 'TenantIndustry', 'TenantNAICS', 'Market', 'Submarket', 'Address.DeliveryAddress', 'Suite', 'Address.Locality', 'Address.County', 'Address.PostalCode', 'Address.RegionName', 'Address.Subdivision', 'Address.Country', 'IsVerified.RawValue', 'SpaceUse', 'LeaseSource', 'LeaseOrigin', 'full_address_std', 'match', 'matched_address', 'latitude', 'longitude', 'state_fips', 'county_fips', 'census_tract', 'census_block', 'geoid', 'tie_matched_addresses', 'tie_latitudes', 'tie_longitudes', 'tie_geoids']


In [82]:
def normalize_zip(value):
    """Return 5-digit ZIP string or None."""
    if pd.isna(value):
        return None

    s = str(value).strip()
    if s == "":
        return None

    # Handle floats like 94107.0
    if re.fullmatch(r"\d+\.0", s):
        s = s.split(".")[0]

    m = re.search(r"(\d{5})", s)
    return m.group(1) if m else None


def extract_zip_from_matched_address(addr):
    """Extract 5-digit ZIP from Census matched_address text."""
    if pd.isna(addr):
        return None

    s = str(addr)
    # Prefer ZIP at end: '..., ST, 94107' or '..., ST, 94107-1234'
    m_end = re.search(r"(\d{5})(?:-\d{4})?\s*$", s)
    if m_end:
        return m_end.group(1)

    m_any = re.search(r"(\d{5})", s)
    return m_any.group(1) if m_any else None


def extract_zips_from_tie_addresses(tie_addresses):
    """Extract unique 5-digit ZIPs from tie_matched_addresses (pipe-delimited)."""
    if pd.isna(tie_addresses):
        return []

    parts = [p.strip() for p in str(tie_addresses).split("|") if p.strip()]
    zips = []
    for part in parts:
        z = extract_zip_from_matched_address(part)
        if z and z not in zips:
            zips.append(z)

    return zips

In [83]:
df["zip_original"] = df["Address.PostalCode"].apply(normalize_zip)
df["zip_geocoder"] = df["matched_address"].apply(extract_zip_from_matched_address)
df["zip_geocoder_tie_list"] = df["tie_matched_addresses"].apply(extract_zips_from_tie_addresses)


def tie_address_count(value):
    if pd.isna(value):
        return 0
    return len([p.strip() for p in str(value).split("|") if p.strip()])


def parse_pipe_list(value):
    if pd.isna(value):
        return []
    return [p.strip() for p in str(value).split("|") if p.strip()]


def to_float_or_none(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def collect_all_geocoder_zips(row):
    zips = list(row["zip_geocoder_tie_list"])
    primary_zip = row["zip_geocoder"]
    if primary_zip and primary_zip not in zips:
        zips.insert(0, primary_zip)
    return zips


def get_matching_tie_output_pairs(row):
    original_zip = row["zip_original"]
    if original_zip is None:
        return []

    tie_addresses = parse_pipe_list(row["tie_matched_addresses"])
    tie_lats = parse_pipe_list(row["tie_latitudes"])
    tie_longs = parse_pipe_list(row["tie_longitudes"])

    n = min(len(tie_addresses), len(tie_lats), len(tie_longs))
    matches = []
    for i in range(n):
        candidate_zip = extract_zip_from_matched_address(tie_addresses[i])
        if candidate_zip == original_zip:
            matches.append((tie_lats[i], tie_longs[i]))

    return matches


df["zip_geocoder_all"] = df.apply(collect_all_geocoder_zips, axis=1)
df["zip_geocoder_all_str"] = df["zip_geocoder_all"].apply(
    lambda zips: " | ".join(zips) if zips else None
)
df["tie_candidate_count"] = df["tie_matched_addresses"].apply(tie_address_count)
df["tie_zip_candidate_count"] = df["zip_geocoder_all"].apply(len)

# Keep explicit category lists so zero-count categories are still reported.
expected_match_types = ["One-to-One", "One-to-Many"]
expected_zip_match_details = [
    "One-to-One Matched",
    "One-to-One Did Not Match",
    "One-to-Many Matched Exactly One Output",
    "One-to-Many Matched Multiple Outputs",
    "One-to-Many Matched No Output",
    "Geocoder ZIP Missing",
    "Original ZIP Missing",
]

df["match_type"] = df.apply(
    lambda row: "One-to-Many" if (row["match"] == "Tie" or row["tie_candidate_count"] > 1) else "One-to-One",
    axis=1,
)


def matching_output_count(row):
    original_zip = row["zip_original"]
    if original_zip is None:
        return 0

    if row["match_type"] == "One-to-One":
        return 1 if row["zip_geocoder"] == original_zip else 0

    tie_matches = get_matching_tie_output_pairs(row)
    if len(tie_matches) > 0:
        return len(tie_matches)

    # Fallback for malformed tie rows where tie fields are missing.
    return 1 if row["zip_geocoder"] == original_zip else 0


df["matching_output_count"] = df.apply(matching_output_count, axis=1)
df["candidate_output_count"] = df.apply(
    lambda row: row["tie_candidate_count"] if row["match_type"] == "One-to-Many" else (1 if row["zip_geocoder"] is not None else 0),
    axis=1,
)


def zip_match_detail(row):
    if row["zip_original"] is None:
        return "Original ZIP Missing"
    if row["candidate_output_count"] == 0:
        return "Geocoder ZIP Missing"

    if row["match_type"] == "One-to-One":
        if row["matching_output_count"] == 1:
            return "One-to-One Matched"
        return "One-to-One Did Not Match"

    if row["matching_output_count"] == 1:
        return "One-to-Many Matched Exactly One Output"
    if row["matching_output_count"] > 1:
        return "One-to-Many Matched Multiple Outputs"
    return "One-to-Many Matched No Output"


def resolve_final_lat_long(row):
    detail = row["zip_match_detail"]

    if detail == "One-to-One Matched":
        return pd.Series([row["latitude"], row["longitude"]])

    if detail == "One-to-Many Matched Exactly One Output":
        tie_matches = get_matching_tie_output_pairs(row)
        if len(tie_matches) == 1:
            return pd.Series([
                to_float_or_none(tie_matches[0][0]),
                to_float_or_none(tie_matches[0][1]),
            ])
        return pd.Series([row["latitude"], row["longitude"]])

    if detail == "One-to-Many Matched Multiple Outputs":
        tie_matches = get_matching_tie_output_pairs(row)
        multi_lat = "MULTIPLE_MATCHING_OUTPUTS: " + " | ".join([m[0] for m in tie_matches])
        multi_long = "MULTIPLE_MATCHING_OUTPUTS: " + " | ".join([m[1] for m in tie_matches])
        return pd.Series([multi_lat, multi_long])

    return pd.Series([None, None])


df["zip_match_detail"] = df.apply(zip_match_detail, axis=1)

# Flag only true non-match outcomes.
flag_details = {
    "One-to-One Did Not Match",
    "One-to-Many Matched No Output",
}
df["flag"] = df["zip_match_detail"].isin(flag_details)
df["flag_detail"] = df["zip_match_detail"].where(df["flag"], None)

# Backward-compatible status columns derived from new detail labels.
detail_to_status = {
    "One-to-One Matched": "Match",
    "One-to-One Did Not Match": "No Match",
    "One-to-Many Matched Exactly One Output": "Match via Tie",
    "One-to-Many Matched Multiple Outputs": "Match via Tie",
    "One-to-Many Matched No Output": "No Match via Tie",
    "Original ZIP Missing": "Original Missing",
    "Geocoder ZIP Missing": "Geocoder Missing",
}
df["zipcode_match_status"] = df["zip_match_detail"].map(detail_to_status)
df["zipcode_match_bool"] = df["zip_match_detail"].isin([
    "One-to-One Matched",
    "One-to-Many Matched Exactly One Output",
    "One-to-Many Matched Multiple Outputs",
])

df[["final_lat", "final_long"]] = df.apply(resolve_final_lat_long, axis=1)

match_type_counts = (
    pd.Series(0, index=expected_match_types, name="rows")
    .add(df["match_type"].value_counts(), fill_value=0)
    .astype(int)
)
zip_match_detail_counts = (
    pd.Series(0, index=expected_zip_match_details, name="rows")
    .add(df["zip_match_detail"].value_counts(), fill_value=0)
    .astype(int)
)

print(match_type_counts)
print("\nZIP match detail counts:")
print(zip_match_detail_counts)
print("\nFlag counts:")
print(df["flag"].value_counts(dropna=False))
print("\nRows with MULTIPLE_MATCHING_OUTPUTS in final_lat:")
print(df["final_lat"].astype(str).str.startswith("MULTIPLE_MATCHING_OUTPUTS:").sum())

One-to-One     9334
One-to-Many      18
dtype: int64

ZIP match detail counts:
Geocoder ZIP Missing                       354
One-to-Many Matched Exactly One Output       0
One-to-Many Matched Multiple Outputs        12
One-to-Many Matched No Output                6
One-to-One Did Not Match                    42
One-to-One Matched                        8937
Original ZIP Missing                         1
dtype: int64

Flag counts:
flag
False    9304
True       48
Name: count, dtype: int64

Rows with MULTIPLE_MATCHING_OUTPUTS in final_lat:
12


In [84]:
summary = (
    df.groupby(["match_type", "zip_match_detail"], dropna=False)
      .size()
      .reset_index(name="rows")
)

# Keep only valid type-detail combinations; these are meaningful zeros.
valid_details_by_type = {
    "One-to-One": [
        "One-to-One Matched",
        "One-to-One Did Not Match",
        "Geocoder ZIP Missing",
        "Original ZIP Missing",
    ],
    "One-to-Many": [
        "One-to-Many Matched Exactly One Output",
        "One-to-Many Matched Multiple Outputs",
        "One-to-Many Matched No Output",
        "Geocoder ZIP Missing",
        "Original ZIP Missing",
    ],
}
all_type_detail = pd.DataFrame(
    [
        {"match_type": t, "zip_match_detail": d}
        for t, details in valid_details_by_type.items()
        for d in details
    ]
)
summary = (
    all_type_detail
    .merge(summary, on=["match_type", "zip_match_detail"], how="left")
    .fillna({"rows": 0})
)
summary["rows"] = summary["rows"].astype(int)

# For original geocoder match, show observed rows only (no structural zero expansion).
match_for_summary = df["match"].fillna("").replace("", "(blank)")
summary_by_match = (
    pd.DataFrame({"match": match_for_summary, "zip_match_detail": df["zip_match_detail"]})
      .groupby(["match", "zip_match_detail"], dropna=False)
      .size()
      .reset_index(name="rows")
      .sort_values(["match", "rows"], ascending=[True, False])
)

summary

,match_type,zip_match_detail,rows
0,One-to-One,One-to-One Matched,8937
1,One-to-One,One-to-One Did Not Match,42
2,One-to-One,Geocoder ZIP Missing,354
3,One-to-One,Original ZIP Missing,1
4,One-to-Many,One-to-Many Matched Exactly One Output,0
5,One-to-Many,One-to-Many Matched Multiple Outputs,12
6,One-to-Many,One-to-Many Matched No Output,6
7,One-to-Many,Geocoder ZIP Missing,0
8,One-to-Many,Original ZIP Missing,0


In [85]:
df[[
    "PropertyId",
    "match",
    "match_type",
    "Address.PostalCode",
    "zip_original",
    "matched_address",
    "tie_matched_addresses",
    "zip_geocoder",
    "zip_geocoder_all_str",
    "tie_candidate_count",
    "matching_output_count",
    "zip_match_detail",
    "flag",
    "flag_detail",
    "final_lat",
    "final_long",
    "census_tract",
    "zipcode_match_status",
    "zipcode_match_bool",
]].head(20)

,PropertyId,match,match_type,Address.PostalCode,zip_original,matched_address,tie_matched_addresses,zip_geocoder,zip_geocoder_all_str,tie_candidate_count,matching_output_count,zip_match_detail,flag,flag_detail,final_lat,final_long,census_tract,zipcode_match_status,zipcode_match_bool
0,36156.0,Match,One-to-One,94108.0,94108,"445 BUSH ST, SAN FRANCISCO, CA, 94108",NaN,94108,94108,0,1,One-to-One Matched,False,None,37.790605,-122.404826,11700.0,Match,True
1,36157.0,Match,One-to-One,94105.0,94105,"33 NEW MONTGOMERY ST, SAN FRANCISCO, CA, 94105",NaN,94105,94105,0,1,One-to-One Matched,False,None,37.788386,-122.401551,61501.0,Match,True
2,36158.0,Match,One-to-One,94104.0,94104,"115 SANSOME ST, SAN FRANCISCO, CA, 94104",NaN,94104,94104,0,1,One-to-One Matched,False,None,37.79134,-122.40085,11700.0,Match,True
3,36159.0,Match,One-to-One,94107.0,94107,"501 2ND ST, SAN FRANCISCO, CA, 94107",NaN,94107,94107,0,1,One-to-One Matched,False,None,37.782822,-122.393193,61508.0,Match,True
4,36160.0,Match,One-to-One,94104.0,94104,"110 SUTTER ST, SAN FRANCISCO, CA, 94104",NaN,94104,94104,0,1,One-to-One Matched,False,None,37.790032,-122.402722,11700.0,Match,True
5,36161.0,Match,One-to-One,94103.0,94103,"1033 MARKET ST, SAN FRANCISCO, CA, 94103",NaN,94103,94103,0,1,One-to-One Matched,False,None,37.781543,-122.410915,17604.0,Match,True
6,36162.0,No_Match,One-to-One,94102.0,94102,NaN,NaN,None,None,0,0,Geocoder ZIP Missing,False,None,None,None,NaN,Geocoder Missing,False
7,36163.0,Match,One-to-One,94108.0,94108,"327 GRANT AVE, SAN FRANCISCO, CA, 94108",NaN,94108,94108,0,1,One-to-One Matched,False,None,37.790124,-122.405583,11700.0,Match,True
8,36164.0,Match,One-to-One,94124.0,94124,"50 MENDELL ST, SAN FRANCISCO, CA, 94124",NaN,94124,94124,0,1,One-to-One Matched,False,None,37.743513,-122.383498,980900.0,Match,True
9,36165.0,Match,One-to-One,94103.0,94103,"1155 MISSION ST, SAN FRANCISCO, CA, 94103",NaN,94103,94103,0,1,One-to-One Matched,False,None,37.778224,-122.412088,17603.0,Match,True


In [86]:
out_path = "../../data/1_derived/4_sf_zipcode_matcher.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")

Saved 9,352 rows to ../../data/1_derived/4_sf_zipcode_matcher.csv


In [87]:
import os
from IPython.display import Markdown, display


def dataframe_to_md_table(frame):
    """Convert a DataFrame to a markdown table without extra dependencies."""
    if frame.empty:
        return "_No rows to display._"

    cols = [str(c) for c in frame.columns]
    header = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join(["---"] * len(cols)) + " |"

    rows = []
    for _, row in frame.iterrows():
        vals = []
        for c in frame.columns:
            v = row[c]
            if pd.isna(v):
                vals.append("")
            else:
                vals.append(str(v).replace("|", "\\|"))
        rows.append("| " + " | ".join(vals) + " |")

    return "\n".join([header, sep] + rows)


status_counts = (
    pd.DataFrame({"zip_match_detail": expected_zip_match_details})
    .merge(
        df["zip_match_detail"].value_counts(dropna=False).rename_axis("zip_match_detail").reset_index(name="rows"),
        on="zip_match_detail",
        how="left",
    )
    .fillna({"rows": 0})
)
status_counts["rows"] = status_counts["rows"].astype(int)
status_counts["pct"] = (status_counts["rows"] / len(df) * 100).round(2)

match_type_counts = (
    pd.DataFrame({"match_type": expected_match_types})
    .merge(
        df["match_type"].value_counts(dropna=False).rename_axis("match_type").reset_index(name="rows"),
        on="match_type",
        how="left",
    )
    .fillna({"rows": 0})
)
match_type_counts["rows"] = match_type_counts["rows"].astype(int)
match_type_counts["pct"] = (match_type_counts["rows"] / len(df) * 100).round(2)

flag_counts = (
    df.groupby(["flag", "flag_detail"], dropna=False)
      .size()
      .reset_index(name="rows")
      .sort_values(["flag", "rows"], ascending=[False, False])
)

mismatch_preview = df.loc[
    df["flag"],
    [
        "PropertyId",
        "match",
        "match_type",
        "Address.PostalCode",
        "matched_address",
        "tie_matched_addresses",
        "zip_original",
        "zip_geocoder_all_str",
        "tie_candidate_count",
        "matching_output_count",
        "zip_match_detail",
        "flag",
        "flag_detail",
        "final_lat",
        "final_long",
        "census_tract",
    ],
].head(25)

one_to_many_exactly_one_preview = df.loc[
    df["zip_match_detail"] == "One-to-Many Matched Exactly One Output",
    [
        "PropertyId",
        "Address.PostalCode",
        "matched_address",
        "tie_matched_addresses",
        "zip_original",
        "zip_geocoder_all_str",
        "tie_candidate_count",
        "matching_output_count",
        "final_lat",
        "final_long",
    ],
].head(25)

one_to_many_multiple_preview = df.loc[
    df["zip_match_detail"] == "One-to-Many Matched Multiple Outputs",
    [
        "PropertyId",
        "Address.PostalCode",
        "tie_matched_addresses",
        "zip_original",
        "matching_output_count",
        "final_lat",
        "final_long",
    ],
].head(25)

report_lines = [
    "# SF ZIP Code Match Report",
    "",
    f"- Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"- Input file: {in_path}",
    f"- Total rows: {len(df):,}",
    "- `match_type` defines relationship shape: One-to-One vs One-to-Many.",
    "- `zip_match_detail` defines exact ZIP comparison outcome within each shape.",
    "- `flag` is True only for non-match outcomes (One-to-One Did Not Match and One-to-Many Matched No Output).",
    "- All expected variations are shown; missing ones are reported as 0.",
    "",
    "## Match Type Counts",
    dataframe_to_md_table(match_type_counts),
    "",
    "## ZIP Match Detail Counts",
    dataframe_to_md_table(status_counts),
    "",
    "## Flag Counts",
    dataframe_to_md_table(flag_counts),
    "",
    "## Match Type x ZIP Detail",
    dataframe_to_md_table(summary),
    "",
    "## Original Geocoder Match x ZIP Detail",
    dataframe_to_md_table(summary_by_match),
    "",
    "## Sample Flagged Rows (Top 25)",
    dataframe_to_md_table(mismatch_preview),
    "",
    "## One-to-Many: Matched Exactly One Output (Top 25)",
    dataframe_to_md_table(one_to_many_exactly_one_preview),
    "",
    "## One-to-Many: Matched Multiple Outputs (Top 25)",
    dataframe_to_md_table(one_to_many_multiple_preview),
]

report_md = "\n".join(report_lines)

display(Markdown(report_md))

md_out_path = "../../output/2_analysis/tables/4_sf_zipcode_matcher_report.md"
os.makedirs(os.path.dirname(md_out_path), exist_ok=True)
with open(md_out_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Saved markdown report to {md_out_path}")

# SF ZIP Code Match Report

- Generated: 2026-03-30 04:52:39
- Input file: ../../data/1_derived/3_sf_properties_census_geocoded.csv
- Total rows: 9,352
- `match_type` defines relationship shape: One-to-One vs One-to-Many.
- `zip_match_detail` defines exact ZIP comparison outcome within each shape.
- `flag` is True only for non-match outcomes (One-to-One Did Not Match and One-to-Many Matched No Output).
- All expected variations are shown; missing ones are reported as 0.

## Match Type Counts
| match_type | rows | pct |
| --- | --- | --- |
| One-to-One | 9334 | 99.81 |
| One-to-Many | 18 | 0.19 |

## ZIP Match Detail Counts
| zip_match_detail | rows | pct |
| --- | --- | --- |
| One-to-One Matched | 8937 | 95.56 |
| One-to-One Did Not Match | 42 | 0.45 |
| One-to-Many Matched Exactly One Output | 0 | 0.0 |
| One-to-Many Matched Multiple Outputs | 12 | 0.13 |
| One-to-Many Matched No Output | 6 | 0.06 |
| Geocoder ZIP Missing | 354 | 3.79 |
| Original ZIP Missing | 1 | 0.01 |

## Flag Counts
| flag | flag_detail | rows |
| --- | --- | --- |
| True | One-to-One Did Not Match | 42 |
| True | One-to-Many Matched No Output | 6 |
| False |  | 9304 |

## Match Type x ZIP Detail
| match_type | zip_match_detail | rows |
| --- | --- | --- |
| One-to-One | One-to-One Matched | 8937 |
| One-to-One | One-to-One Did Not Match | 42 |
| One-to-One | Geocoder ZIP Missing | 354 |
| One-to-One | Original ZIP Missing | 1 |
| One-to-Many | One-to-Many Matched Exactly One Output | 0 |
| One-to-Many | One-to-Many Matched Multiple Outputs | 12 |
| One-to-Many | One-to-Many Matched No Output | 6 |
| One-to-Many | Geocoder ZIP Missing | 0 |
| One-to-Many | Original ZIP Missing | 0 |

## Original Geocoder Match x ZIP Detail
| match | zip_match_detail | rows |
| --- | --- | --- |
| (blank) | Geocoder ZIP Missing | 1 |
| Match | One-to-One Matched | 8937 |
| Match | One-to-One Did Not Match | 42 |
| Match | Original ZIP Missing | 1 |
| No_Match | Geocoder ZIP Missing | 353 |
| Tie | One-to-Many Matched Multiple Outputs | 12 |
| Tie | One-to-Many Matched No Output | 6 |

## Sample Flagged Rows (Top 25)
| PropertyId | match | match_type | Address.PostalCode | matched_address | tie_matched_addresses | zip_original | zip_geocoder_all_str | tie_candidate_count | matching_output_count | zip_match_detail | flag | flag_detail | final_lat | final_long | census_tract |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 320469.0 | Match | One-to-One | 94111.0 | 1 THE EMBARCADERO, SAN FRANCISCO, CA, 94107 |  | 94111 | 94107 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 61508.0 |
| 320471.0 | Match | One-to-One | 94111.0 | 3 THE EMBARCADERO, SAN FRANCISCO, CA, 94107 |  | 94111 | 94107 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 61508.0 |
| 322770.0 | Match | One-to-One | 94103.0 | 245 VAN NESS AVE, SAN FRANCISCO, CA, 94102 |  | 94103 | 94102 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 16200.0 |
| 323170.0 | Match | One-to-One | 94133.0 | 2 N POINT ST, SAN FRANCISCO, CA, 94109 |  | 94133 | 94109 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 10101.0 |
| 334971.0 | Match | One-to-One | 94080.0 | 1265 MISSION ST, SAN FRANCISCO, CA, 94103 |  | 94080 | 94103 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 17603.0 |
| 365252.0 | Tie | One-to-Many | 94105.0 | 26 PIER, SAN FRANCISCO, CA, 94111 | 26 PIER, SAN FRANCISCO, CA, 94111 \| 26 PIER, SAN FRANCISCO, CA, 94124 | 94105 | 94111 \| 94124 | 2 | 0 | One-to-Many Matched No Output | True | One-to-Many Matched No Output |  |  | 10500.0 |
| 610198.0 | Match | One-to-One | 94129.0 | 38 KEYES ALY, SAN FRANCISCO, CA, 94133 |  | 94129 | 94133 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 10701.0 |
| 683604.0 | Match | One-to-One | 94129.0 | 9 FUNSTON AVE, SAN FRANCISCO, CA, 94118 |  | 94129 | 94118 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 40200.0 |
| 683606.0 | Match | One-to-One | 94129.0 | 10 FUNSTON AVE, SAN FRANCISCO, CA, 94118 |  | 94129 | 94118 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 40200.0 |
| 704054.0 | Match | One-to-One | 94102.0 | 108 TAYLOR RD, SAN FRANCISCO, CA, 94129 |  | 94102 | 94129 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 60100.0 |
| 744326.0 | Match | One-to-One | 94129.0 | 211 LINCOLN WAY, SAN FRANCISCO, CA, 94122 |  | 94129 | 94122 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 30101.0 |
| 745372.0 | Match | One-to-One | 94129.0 | 1185 MASON ST, SAN FRANCISCO, CA, 94108 |  | 94129 | 94108 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 11200.0 |
| 745373.0 | Match | One-to-One | 94129.0 | 1187 MASON ST, SAN FRANCISCO, CA, 94108 |  | 94129 | 94108 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 11200.0 |
| 760583.0 | Match | One-to-One | 94109.0 | 1000 S VAN NESS AVE, SAN FRANCISCO, CA, 94110 |  | 94109 | 94110 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 20801.0 |
| 768159.0 | Match | One-to-One | 94129.0 | 222 HALLECK ST, SAN FRANCISCO, CA, 94104 |  | 94129 | 94104 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 11700.0 |
| 778285.0 | Match | One-to-One | 94111.0 | 5 THE EMBARCADERO, SAN FRANCISCO, CA, 94107 |  | 94111 | 94107 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 61508.0 |
| 782910.0 | Match | One-to-One | 94129.0 | 7 FUNSTON AVE, SAN FRANCISCO, CA, 94118 |  | 94129 | 94118 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 40200.0 |
| 786231.0 | Match | One-to-One | 94129.0 | 100 MONTGOMERY ST, SAN FRANCISCO, CA, 94104 |  | 94129 | 94104 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 11700.0 |
| 844850.0 | Match | One-to-One | 94129.0 | 215 LINCOLN WAY, SAN FRANCISCO, CA, 94122 |  | 94129 | 94122 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 30101.0 |
| 850617.0 | Match | One-to-One | 94015.0 | 331 WESTLAKE AVE, DALY CITY, CA, 94014 |  | 94015 | 94014 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 600600.0 |
| 1352783.0 | Match | One-to-One | 94105.0 | 3 JACKSON ST, SAN FRANCISCO, CA, 94111 |  | 94105 | 94111 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 10500.0 |
| 1352787.0 | Tie | One-to-Many | 94105.0 | 5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 | 5 PACIFIC AVE, SAN FRANCISCO, CA, 94111 \| 5 PACIFIC AVE, SAN FRANCISCO, CA, 94133 | 94105 | 94111 \| 94133 | 2 | 0 | One-to-Many Matched No Output | True | One-to-Many Matched No Output |  |  | 10500.0 |
| 5580579.0 | Match | One-to-One | 94101.0 | 2381 CHESTNUT ST, SAN FRANCISCO, CA, 94123 |  | 94101 | 94123 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 12801.0 |
| 6331737.0 | Match | One-to-One | 94015.0 | 415 WESTLAKE AVE, DALY CITY, CA, 94014 |  | 94015 | 94014 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 600600.0 |
| 6671033.0 | Match | One-to-One | 94015.0 | 406 WESTLAKE AVE, DALY CITY, CA, 94014 |  | 94015 | 94014 | 0 | 0 | One-to-One Did Not Match | True | One-to-One Did Not Match |  |  | 600600.0 |

## One-to-Many: Matched Exactly One Output (Top 25)
_No rows to display._

## One-to-Many: Matched Multiple Outputs (Top 25)
| PropertyId | Address.PostalCode | tie_matched_addresses | zip_original | matching_output_count | final_lat | final_long |
| --- | --- | --- | --- | --- | --- | --- |
| 321580.0 | 94124.0 | 1485 BAY SHORE BLVD, SAN FRANCISCO, CA, 94124 \| 1485 BAYSHORE BLVD, SAN FRANCISCO, CA, 94124 | 94124 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.725535736384 \| 37.725535736384 | MULTIPLE_MATCHING_OUTPUTS: -122.401401989518 \| -122.401401989518 |
| 333689.0 | 94403.0 | 3130 LA SELVA CIR, SAN MATEO, CA, 94403 \| 3130 LA SELVA ST, SAN MATEO, CA, 94403 | 94403 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.542678745951 \| 37.54307237241 | MULTIPLE_MATCHING_OUTPUTS: -122.284437924233 \| -122.285084558557 |
| 783773.0 | 94401.0 | 256 S EL CAMINO REAL, SAN MATEO, CA, 94401 \| 256 N EL CAMINO REAL, SAN MATEO, CA, 94401 | 94401 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.563454197511 \| 37.568959419958 | MULTIPLE_MATCHING_OUTPUTS: -122.326559076275 \| -122.333189329068 |
| 845493.0 | 94015.0 | 100 SKYLINE BLVD, DALY CITY, CA, 94015 \| 100 SKYLINE DR, DALY CITY, CA, 94015 | 94015 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.673925650836 \| 37.691764185401 | MULTIPLE_MATCHING_OUTPUTS: -122.488596769759 \| -122.494884037259 |
| 4318893.0 | 94133.0 | 435 BROADWAY ST, SAN FRANCISCO, CA, 94133 \| 435 BROADWAY, SAN FRANCISCO, CA, 94133 | 94133 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.798087228536 \| 37.798087228536 | MULTIPLE_MATCHING_OUTPUTS: -122.404443200708 \| -122.404443200708 |
| 4327803.0 | 94005.0 | 88 S HILL DR, BRISBANE, CA, 94005 \| 88 W HILL DR, BRISBANE, CA, 94005 | 94005 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.686556712518 \| 37.692860513616 | MULTIPLE_MATCHING_OUTPUTS: -122.411619079175 \| -122.421788578945 |
| 4570159.0 | 94401.0 | 313 S SAN MATEO DR, SAN MATEO, CA, 94401 \| 313 N SAN MATEO DR, SAN MATEO, CA, 94401 | 94401 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.564197193327 \| 37.571051391926 | MULTIPLE_MATCHING_OUTPUTS: -122.323970606746 \| -122.331329802994 |
| 5027545.0 | 94066.0 | 751 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 751 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 94066 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.628528342383 \| 37.630969151044 | MULTIPLE_MATCHING_OUTPUTS: -122.417215794707 \| -122.406114264877 |
| 5034363.0 | 94066.0 | 777 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 777 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 94066 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.628498597917 \| 37.6309735587 | MULTIPLE_MATCHING_OUTPUTS: -122.417293460825 \| -122.405808919857 |
| 5344465.0 | 94066.0 | 675 SAN BRUNO AVE W, SAN BRUNO, CA, 94066 \| 675 SAN BRUNO AVE E, SAN BRUNO, CA, 94066 | 94066 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.628849658651 \| 37.630955198054 | MULTIPLE_MATCHING_OUTPUTS: -122.416374331022 \| -122.407031937646 |
| 6395899.0 | 94015.0 | 31 SKYLINE BLVD, DALY CITY, CA, 94015 \| 31 SKYLINE DR, DALY CITY, CA, 94015 | 94015 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.677628487762 \| 37.693853937592 | MULTIPLE_MATCHING_OUTPUTS: -122.48994771527 \| -122.495688149313 |
| 8930622.0 | 94063.0 | 499 SEAPORT BLVD, REDWOOD CITY, CA, 94063 \| 499 SEAPORT CT, REDWOOD CITY, CA, 94063 | 94063 | 2 | MULTIPLE_MATCHING_OUTPUTS: 37.49849564196 \| 37.50297070501 | MULTIPLE_MATCHING_OUTPUTS: -122.212266111685 \| -122.211757491611 |

Saved markdown report to ../../output/2_analysis/tables/4_sf_zipcode_matcher_report.md
